# 1. Setup and document selection


In [1]:
import importlib
import json
import sys
import unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display

cwd=Path.cwd().resolve()
candidates=[cwd,*cwd.parents,cwd/"SheCraft"/"ai-service",cwd/"ai-service"]
PROJECT_ROOT=next((x for x in candidates if (x/"src"/"rag"/"scripts").is_dir() and (x/"rag_documents").is_dir()),None)
if PROJECT_ROOT is None: raise RuntimeError("Could not locate ai-service project root.")
RAG_SRC_DIR=PROJECT_ROOT/"src"/"rag"
if str(RAG_SRC_DIR) not in sys.path: sys.path.insert(0,str(RAG_SRC_DIR))
DOCUMENTS_DIR=PROJECT_ROOT/"rag_documents"
ARTIFACT_ROOT=PROJECT_ROOT/"data"/"rag"/"processed_documents"
EVALUATION_DIR=PROJECT_ROOT/"data"/"rag"/"evaluations"
PDF_FILES=sorted(DOCUMENTS_DIR.rglob("*.pdf"))
if not PDF_FILES: raise FileNotFoundError(f"No PDFs found in {DOCUMENTS_DIR}")
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
FORCE_REEXTRACTION=False
print("PDFs selected:",len(PDF_FILES),"device:",DEVICE,"cache:",ARTIFACT_ROOT)

PDFs selected: 39 device: cuda cache: C:\Users\User\Desktop\LLMs - Samer Saab\Project\Jewelry\SheCraft\ai-service\data\rag\processed_documents


# 2. Chunking strategies


In [2]:
STRATEGY_CONFIGS = {
    "fixed_size": {
        "enabled": True,
        "kwargs": {"chunk_size": 800, "overlap": 100},
    },
    "recursive": {
        "enabled": True,
        "kwargs": {"chunk_size": 800, "overlap": 100},
    },
    "sentence": {
        "enabled": True,
        "kwargs": {"sentences_per_chunk": 4, "sentence_overlap": 1},
    },
    "structure_aware": {
        "enabled": True,
        "kwargs": {"chunk_size": 800, "overlap": 100},
    },
    "semantic": {
        "enabled": True,
        "kwargs": {
            "similarity_threshold": 0.65,
            "max_chunk_size": 800,
            "unit_prefix": "passage: ",
        },
    },
}

enabled_strategies = {
    name: config["kwargs"]
    for name, config in STRATEGY_CONFIGS.items()
    if config["enabled"]
}

print("Methods that will run:")
for name, settings in enabled_strategies.items():
    print(f" - {name}: {settings}")


Methods that will run:
 - fixed_size: {'chunk_size': 800, 'overlap': 100}
 - recursive: {'chunk_size': 800, 'overlap': 100}
 - sentence: {'sentences_per_chunk': 4, 'sentence_overlap': 1}
 - structure_aware: {'chunk_size': 800, 'overlap': 100}
 - semantic: {'similarity_threshold': 0.65, 'max_chunk_size': 800, 'unit_prefix': 'passage: '}


# 3. Import methods from `scripts` and load the embedding model


In [3]:
pipeline_module=importlib.reload(importlib.import_module("scripts.pipeline"))
extraction_module=importlib.reload(importlib.import_module("scripts.2_extraction"))
chunking_module=importlib.reload(importlib.import_module("scripts.4_chunking"))
embeddings_module=importlib.reload(importlib.import_module("scripts.5_embeddings"))
extract_pdf=extraction_module.extract_pdf
chunk_extracted_documents=chunking_module.chunk_extracted_documents
load_embedding_model=embeddings_module.load_embedding_model
prepare_texts=embeddings_module.prepare_texts
embed_texts=embeddings_module.embed_texts
embed_queries=embeddings_module.embed_queries
cosine_search=embeddings_module.cosine_search
EMBEDDING_BATCH_SIZE=8 if DEVICE=="cuda" else 2
RETRIEVAL_MODEL_NAME="intfloat/multilingual-e5-base"
CHUNKING_MODEL_NAME=RETRIEVAL_MODEL_NAME
model_kwargs={"torch_dtype":torch.float16} if DEVICE=="cuda" else None
retrieval_model=load_embedding_model(RETRIEVAL_MODEL_NAME,device=DEVICE,local_files_only=True,model_kwargs=model_kwargs)
chunking_model=load_embedding_model(CHUNKING_MODEL_NAME,device=DEVICE,local_files_only=True,model_kwargs=model_kwargs)
retrieval_model.max_seq_length=embeddings_module.DEFAULT_MAX_SEQUENCE_LENGTH
chunking_model.max_seq_length=embeddings_module.DEFAULT_MAX_SEQUENCE_LENGTH
embedding_model=retrieval_model
EMBEDDING_MODEL_NAME=RETRIEVAL_MODEL_NAME
print("Chunking/retrieval model:",RETRIEVAL_MODEL_NAME,"device:",DEVICE)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Chunking/retrieval model: intfloat/multilingual-e5-base device: cuda


# 4. Extract text from every selected PDF


In [4]:
pipeline_result=pipeline_module.prepare_documents_for_rag(
    PDF_FILES, recursive=False, artifact_root=ARTIFACT_ROOT,
    chunking_strategy="structure_aware", chunking_options={"chunk_size":800,"overlap":100},
    extract_images=False, extraction_options={"use_ocr_fallback":True}, force=FORCE_REEXTRACTION,
)
extracted_documents=pipeline_result["extracted_documents"]
extraction_summary=pd.DataFrame([
    {"file_name":doc["file_name"],"pages":doc["page_count"],
     "pages_with_text":sum(bool(page["final_text"].strip()) for page in doc["pages"]),
     "final_text_characters":sum(len(page["final_text"]) for page in doc["pages"])}
    for doc in extracted_documents
])
display(extraction_summary)
print("Cached PDF extraction reused where available.")

,file_name,pages,pages_with_text,final_text_characters
0,birthstone_guide.pdf,5,5,5231
1,engagement_ring_settings_guide.pdf,15,14,19066
2,alexandrite.pdf,3,3,2422
3,amber.pdf,2,2,2524
4,amethyst.pdf,3,2,2604
5,ametrine.pdf,2,2,2207
6,aquamarine.pdf,2,2,2525
7,citrine.pdf,2,2,2557
8,diamond.pdf,2,2,2512
9,emerald.pdf,2,2,2599


Cached PDF extraction reused where available.


## 5. Generate chunks with every enabled method


In [5]:
chunks_by_strategy = {}

for strategy_name, strategy_kwargs in enabled_strategies.items():
    print(f"Chunking with {strategy_name}: {strategy_kwargs}")
    runtime_kwargs = dict(strategy_kwargs)
    if strategy_name == "semantic":
        runtime_kwargs["model"] = chunking_model

    chunks_by_strategy[strategy_name] = chunk_extracted_documents(
        extracted_documents,
        strategy=strategy_name,
        **runtime_kwargs,
    )
    print("  chunks created:", len(chunks_by_strategy[strategy_name]))


Chunking with fixed_size: {'chunk_size': 800, 'overlap': 100}
  chunks created: 298
Chunking with recursive: {'chunk_size': 800, 'overlap': 100}
  chunks created: 297
Chunking with sentence: {'sentences_per_chunk': 4, 'sentence_overlap': 1}
  chunks created: 1803
Chunking with structure_aware: {'chunk_size': 800, 'overlap': 100}
  chunks created: 319
Chunking with semantic: {'similarity_threshold': 0.65, 'max_chunk_size': 800, 'unit_prefix': 'passage: '}
  chunks created: 283


## 6. Compare chunk-size distributions


In [6]:
comparison_rows = []

for strategy_name, chunks in chunks_by_strategy.items():
    lengths = chunks["character_count"]
    comparison_rows.append({
        "strategy": strategy_name,
        "chunks": len(chunks),
        "documents": chunks["file_name"].nunique(),
        "pages": chunks[["file_name", "page_number"]].drop_duplicates().shape[0],
        "min_chars": int(lengths.min()),
        "median_chars": int(lengths.median()),
        "mean_chars": round(float(lengths.mean()), 1),
        "p90_chars": int(lengths.quantile(0.90)),
        "max_chars": int(lengths.max()),
        "short_chunks_lt_100": int((lengths < 100).sum()),
        "short_chunk_percent": round(float((lengths < 100).mean() * 100), 1),
    })

strategy_comparison = pd.DataFrame(comparison_rows).set_index("strategy")
display(strategy_comparison)


,chunks,documents,pages,min_chars,median_chars,mean_chars,p90_chars,max_chars,short_chunks_lt_100,short_chunk_percent
strategy,,,,,,,,,,
fixed_size,298,39,165,67,702,589.8,800,800,1,0.3
recursive,297,39,165,67,706,578.9,792,799,2,0.7
sentence,1803,39,165,6,99,114.9,217,335,910,50.5
structure_aware,319,39,165,12,531,512.8,776,800,13,4.1
semantic,283,39,165,6,688,566.8,794,800,12,4.2


## 7. Inspect comparable chunk samples


In [7]:
SAMPLE_FILE_NAME = chunks_by_strategy["structure_aware"].iloc[0]["file_name"]
SAMPLE_PAGE_NUMBER = int(chunks_by_strategy["structure_aware"].iloc[0]["page_number"])
SAMPLE_LIMIT_PER_STRATEGY = 3
for strategy_name,chunks in chunks_by_strategy.items():
    sample=chunks[(chunks["file_name"]==SAMPLE_FILE_NAME)&(chunks["page_number"]==SAMPLE_PAGE_NUMBER)].head(SAMPLE_LIMIT_PER_STRATEGY)
    print("METHOD:",strategy_name,"FILE:",SAMPLE_FILE_NAME,"PAGE:",SAMPLE_PAGE_NUMBER)
    for row in sample.itertuples(index=False):
        print(f"\n[{row.chunk_id}] {row.character_count} characters\n{row.text}")

METHOD: fixed_size FILE: birthstone_guide.pdf PAGE: 1

[birthstone_guide-0cf44a028f83-p0001-c000] 538 characters
FROM THE VAULT
April 2017 – August 2017
Photo: Orasa Weldon/GIA Aquamarine ring, gift of Dr. Keith G. Wurtz; blue topaz earrings, courtesy Brumani; boulder
opal from Queensland, Australia.
EXHIBIT DESCRIPTION
An exhibit of bewitchingly beautiful gems, “GIA Celebrates Birthstones” showcases the diversity of GIA’s
museum collection and unlocks the mysteries behind the meaning and symbolism of birthstones. This
exhibit features more than 250 gems, minerals and jewelry.
file:///C:/Users/User/Downloads/gia_celebrates_birthstones.html
1/5
METHOD: recursive FILE: birthstone_guide.pdf PAGE: 1

[birthstone_guide-0cf44a028f83-p0001-c000] 538 characters
FROM THE VAULT
April 2017 – August 2017
Photo: Orasa Weldon/GIA Aquamarine ring, gift of Dr. Keith G. Wurtz; blue topaz earrings, courtesy Brumani; boulder
opal from Queensland, Australia.
EXHIBIT DESCRIPTION
An exhibit of bewitchingly 

## 8. Integrity checks


In [8]:
integrity_rows = []

for strategy_name, chunks in chunks_by_strategy.items():
    config = enabled_strategies[strategy_name]
    configured_limit = config.get("chunk_size", config.get("max_chunk_size"))
    limit_respected = (
        True
        if configured_limit is None
        else bool(chunks["character_count"].le(configured_limit).all())
    )
    integrity_rows.append({
        "strategy": strategy_name,
        "nonempty_text": bool(chunks["text"].str.strip().ne("").all()),
        "unique_chunk_ids": bool(chunks["chunk_id"].is_unique),
        "valid_page_numbers": bool(chunks["page_number"].ge(1).all()),
        "source_files_exist": bool(
            chunks["source_path"].map(lambda value: Path(value).is_file()).all()
        ),
        "configured_size_limit_respected": limit_respected,
    })

integrity_checks = pd.DataFrame(integrity_rows).set_index("strategy")
display(integrity_checks)
assert integrity_checks.to_numpy().all(), "At least one chunk integrity check failed."
print("All integrity checks passed.")


,nonempty_text,unique_chunk_ids,valid_page_numbers,source_files_exist,configured_size_limit_respected
strategy,,,,,
fixed_size,True,True,True,True,True
recursive,True,True,True,True,True
sentence,True,True,True,True,True
structure_aware,True,True,True,True,True
semantic,True,True,True,True,True


All integrity checks passed.


## 9. Retrieval and answer-keyword evaluation

Load the shared 100-question benchmark. The evaluation reports both expected-document retrieval and answer-keyword coverage within retrieved chunks.


In [9]:
EVALUATION_CASES_FILE=EVALUATION_DIR/"retrieval_method_benchmark_100.json"
EVALUATION_RESULTS_FILE=EVALUATION_DIR/"chunking_method_results.json"
evaluation_cases=json.loads(EVALUATION_CASES_FILE.read_text(encoding="utf-8"))
if len(evaluation_cases)!=100:
    raise ValueError(f"Expected 100 benchmark cases, found {len(evaluation_cases)}")
missing_keywords=[case["id"] for case in evaluation_cases if not case.get("expected_keywords")]
if missing_keywords:
    raise ValueError(f"Cases are missing expected_keywords: {missing_keywords}")

def normalize_keyword_text(value):
    normalized=unicodedata.normalize("NFKD",str(value).casefold())
    return "".join(character for character in normalized if character.isalnum())

# Make sure every answer term is present in the extracted text of a labeled source PDF.
source_text_by_name={doc["file_name"]:" ".join(page["final_text"] for page in doc["pages"]) for doc in extracted_documents}
available_source_names=set(source_text_by_name)
missing_targets=sorted({name for case in evaluation_cases for name in case["expected_documents"]}-available_source_names)
if missing_targets:
    raise FileNotFoundError(f"Benchmark target PDFs are not in the current corpus: {missing_targets}. Update expected_documents or restore those PDFs.")
unverified_terms=[]
for case in evaluation_cases:
    labeled_text=" ".join(source_text_by_name.get(name,"") for name in case["expected_documents"])
    normalized_source=normalize_keyword_text(labeled_text)
    absent=[term for term in case["expected_keywords"] if normalize_keyword_text(term) not in normalized_source]
    if absent: unverified_terms.append((case["id"],absent))
if unverified_terms:
    raise ValueError(f"Some answer keywords are not found in their labeled PDF text: {unverified_terms[:10]}")

query_embeddings=embed_queries(
    [case["query"] for case in evaluation_cases],model=embedding_model,
    model_name=EMBEDDING_MODEL_NAME,batch_size=EMBEDDING_BATCH_SIZE,
)
print("Evaluation cases:",len(evaluation_cases),"languages:",pd.Series([case["language"] for case in evaluation_cases]).value_counts().to_dict())
print("All expected keywords verified in the labeled source text.")
display(pd.DataFrame(evaluation_cases)[["id","query","expected_documents","expected_keywords"]].head(10))


Evaluation cases: 100 languages: {'en': 60, 'ar': 20, 'fr': 20}
All expected keywords verified in the labeled source text.


,id,query,expected_documents,expected_keywords
0,retrieval_001,Should pearl jewelry be cleaned with a steam c...,[pearl_care_cleaning_guide.pdf],"[ultrasonic, steam cleaner, warm soapy water, ..."
1,retrieval_002,"Why do pearls scratch easily, and what routine...",[pearl_care_cleaning_guide.pdf],"[Mohs hardness, scratched or abraded, perfume,..."
2,retrieval_003,How should a jeweler measure a finger for an a...,[Ring Size Guide.pdf],"[jeweler, finger gauges, ring sizer, ring mand..."
3,retrieval_004,How can I find a ring size that fits snugly bu...,[Ring Size Guide.pdf],"[ring size, fits snugly, finger gauges, best c..."
4,retrieval_005,Which ring metals are best for everyday wear a...,[ring_metals_everyday_wear.pdf],"[platinum, gold, titanium, durability]"
5,retrieval_006,"How do white gold, yellow gold, and rose gold ...",[gold_colors_guide.pdf],"[white gold, yellow gold, rose gold, rhodium]"
6,retrieval_007,"What are the differences between solitaire, ha...",[engagement_ring_settings_guide.pdf],"[solitaire, halo, pav?, three-stone]"
7,retrieval_008,How are gemstones secured in common jewelry se...,[jewelry_settings_bench_guide.pdf],"[bezel, prong, pav?, channel]"
8,retrieval_009,What are common jewelry chain types and their ...,[jewelry_chain_guide.pdf],"[cable, curb, figaro, rope]"
9,retrieval_010,How can I style and layer necklaces with diffe...,[necklace_stacking_guide.pdf],"[necklace stack, different lengths, natural se..."


In [10]:
retrieval_rows=[]
strategy_metric_rows=[]
for strategy_name,chunks in chunks_by_strategy.items():
    print(f"Embedding chunks for retrieval evaluation: {strategy_name}")
    prepared_chunks=prepare_texts(chunks["text"].tolist(),model_name=EMBEDDING_MODEL_NAME,text_type="passage")
    chunk_embeddings=embed_texts(prepared_chunks,embedding_model,batch_size=EMBEDDING_BATCH_SIZE)
    scores,positions=cosine_search(query_embeddings,chunk_embeddings,top_k=min(5,len(chunks)))
    document_ranks=[]; answer_ranks=[]; document_hits={1:[],3:[],5:[]}; answer_hits={1:[],3:[],5:[]}
    target_document_coverages=[]; keyword_coverages=[]
    for query_index,case in enumerate(evaluation_cases):
        expected_documents=set(case["expected_documents"])
        expected_keywords=list(case["expected_keywords"])
        normalized_keywords={keyword:normalize_keyword_text(keyword) for keyword in expected_keywords}
        found_documents=set(); found_keywords=set(); first_document_rank=None; first_answer_rank=None; top_results=[]
        for rank,position in enumerate(positions[query_index],start=1):
            row=chunks.iloc[int(position)]
            document_relevant=row["file_name"] in expected_documents
            normalized_chunk=normalize_keyword_text(row["text"])
            matched_keywords=[keyword for keyword,normalized in normalized_keywords.items() if normalized in normalized_chunk]
            answer_relevant=document_relevant and bool(matched_keywords)
            if document_relevant:
                found_documents.add(row["file_name"])
                if first_document_rank is None:first_document_rank=rank
            if answer_relevant:
                found_keywords.update(matched_keywords)
                if first_answer_rank is None:first_answer_rank=rank
            top_results.append({
                "rank":rank,"chunk_id":row["chunk_id"],"file_name":row["file_name"],
                "page_number":int(row["page_number"]),"score":round(float(scores[query_index,rank-1]),6),
                "document_relevant":bool(document_relevant),"answer_relevant":bool(answer_relevant),
                "matched_keywords":matched_keywords,
            })
        document_ranks.append(first_document_rank); answer_ranks.append(first_answer_rank)
        for k in document_hits:document_hits[k].append(first_document_rank is not None and first_document_rank<=k)
        for k in answer_hits:answer_hits[k].append(first_answer_rank is not None and first_answer_rank<=k)
        doc_coverage=len(found_documents)/len(expected_documents) if expected_documents else 0.0
        keyword_coverage=len(found_keywords)/len(expected_keywords) if expected_keywords else 0.0
        target_document_coverages.append(doc_coverage); keyword_coverages.append(keyword_coverage)
        retrieval_rows.append({
            "strategy":strategy_name,"case_id":case["id"],"query":case["query"],
            "language":case["language"],"cross_pdf":case["cross_pdf"],
            "expected_documents":case["expected_documents"],"expected_keywords":expected_keywords,
            "first_document_rank":first_document_rank,"first_answer_rank":first_answer_rank,
            "target_document_coverage_at_5":doc_coverage,"keyword_coverage_at_5":keyword_coverage,
            "matched_keywords_at_5":sorted(found_keywords),"top_5":top_results,
        })
    doc_mrr=float(np.mean([0.0 if rank is None else 1.0/rank for rank in document_ranks]))
    answer_mrr=float(np.mean([0.0 if rank is None else 1.0/rank for rank in answer_ranks]))
    strategy_metric_rows.append({
        "strategy":strategy_name,
        "document_recall_at_1":float(np.mean(document_hits[1])),
        "document_recall_at_3":float(np.mean(document_hits[3])),
        "document_recall_at_5":float(np.mean(document_hits[5])),
        "document_mrr_at_5":doc_mrr,
        "target_document_coverage_at_5":float(np.mean(target_document_coverages)),
        "answer_recall_at_1":float(np.mean(answer_hits[1])),
        "answer_recall_at_3":float(np.mean(answer_hits[3])),
        "answer_recall_at_5":float(np.mean(answer_hits[5])),
        "answer_mrr_at_5":answer_mrr,
        "keyword_coverage_at_5":float(np.mean(keyword_coverages)),
    })
retrieval_details=pd.DataFrame(retrieval_rows)
retrieval_metrics=pd.DataFrame(strategy_metric_rows).set_index("strategy")
display(retrieval_metrics.sort_values(["answer_recall_at_5","keyword_coverage_at_5","document_recall_at_5"],ascending=False))


Embedding chunks for retrieval evaluation: fixed_size


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Embedding chunks for retrieval evaluation: recursive


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Embedding chunks for retrieval evaluation: sentence


Batches:   0%|          | 0/226 [00:00<?, ?it/s]

Embedding chunks for retrieval evaluation: structure_aware


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Embedding chunks for retrieval evaluation: semantic


Batches:   0%|          | 0/36 [00:00<?, ?it/s]

,document_recall_at_1,document_recall_at_3,document_recall_at_5,document_mrr_at_5,target_document_coverage_at_5,answer_recall_at_1,answer_recall_at_3,answer_recall_at_5,answer_mrr_at_5,keyword_coverage_at_5
strategy,,,,,,,,,,
fixed_size,0.75,0.89,0.91,0.815000,0.891667,0.69,0.85,0.89,0.769500,0.79200
recursive,0.76,0.89,0.90,0.824167,0.886667,0.68,0.86,0.88,0.766667,0.79600
semantic,0.79,0.89,0.89,0.830000,0.876667,0.72,0.87,0.88,0.783667,0.77025
structure_aware,0.70,0.87,0.88,0.780833,0.860000,0.64,0.86,0.87,0.742500,0.75100
sentence,0.66,0.84,0.89,0.748667,0.871667,0.53,0.78,0.85,0.654833,0.53825


In [11]:
failed_answer_cases=retrieval_details[retrieval_details["first_answer_rank"].isna()]
display(failed_answer_cases[["strategy","case_id","query","expected_documents","expected_keywords","language"]])
results_payload={"schema_version":2,"embedding_model":EMBEDDING_MODEL_NAME,
 "evaluation_case_file":str(EVALUATION_CASES_FILE.relative_to(PROJECT_ROOT)),
 "strategy_configs":enabled_strategies,
 "retrieval_metrics":retrieval_metrics.reset_index().to_dict(orient="records"),
 "chunk_statistics":strategy_comparison.reset_index().to_dict(orient="records"),
 "case_results":retrieval_rows}
EVALUATION_RESULTS_FILE.parent.mkdir(parents=True,exist_ok=True)
EVALUATION_RESULTS_FILE.write_text(json.dumps(results_payload,ensure_ascii=False,indent=2),encoding="utf-8")
print("Saved evaluation results:",EVALUATION_RESULTS_FILE)


,strategy,case_id,query,expected_documents,expected_keywords,language
46,fixed_size,retrieval_047,How can chain-link styles be combined when cre...,"[jewelry_chain_guide.pdf, necklace_stacking_gu...","[link styles, box chains, cable link, layered ...",en
61,fixed_size,retrieval_062,كيف يصف الدليل حجر الكهرمان من حيث اللون والتر...,[amber.pdf],"[fossilized tree resin, animal and plant inclu...",ar
62,fixed_size,retrieval_063,ما الذي يميّز حجر الجمشت ويجعل لونه أو تأثيره ...,[amethyst.pdf],"[amethyst, quartz, purple, ruby, emerald]",ar
65,fixed_size,retrieval_066,كيف يصف الدليل حجر السترين من حيث اللون والترك...,[citrine.pdf],"[citrine, quartz, yellow, topaz, amethyst]",ar
66,fixed_size,retrieval_067,ما الذي يميّز حجر الألماس ويجعل لونه أو تأثيره...,[diamond.pdf],"[diamond, carbon, hardness, Mohs]",ar
...,...,...,...,...,...,...
471,semantic,retrieval_072,إلى أي مجموعة معدنية ينتمي حجر اليشم وما أبرز ...,[jade.pdf],"[nephrite, jadeite, green omphacite, generic t...",ar
472,semantic,retrieval_073,ما الخصائص المميزة لحجر الكونزيت المذكورة في ا...,[kunzite.pdf],"[kunzite, spodumene, lithium, manganese]",ar
477,semantic,retrieval_078,كيف يصف الدليل حجر اللؤلؤ من حيث اللون والتركي...,[pearl.pdf],"[pearl, nacre, mollusk, organic]",ar
479,semantic,retrieval_080,إلى أي مجموعة معدنية ينتمي حجر الياقوت الأحمر ...,[ruby.pdf],"[ruby, corundum, chromium, red]",ar


Saved evaluation results: C:\Users\User\Desktop\LLMs - Samer Saab\Project\Jewelry\SheCraft\ai-service\data\rag\evaluations\chunking_method_results.json
